In [1]:
from delta import *
from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName("GoldLayer")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "2g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.3.0


In [5]:
BASE = "/home/jovyan/work/delta_lake/silver"

orders       = spark.read.format("delta").load(f"{BASE}/orders")
order_items  = spark.read.format("delta").load(f"{BASE}/order_items")
products     = spark.read.format("delta").load(f"{BASE}/products")
customers    = spark.read.format("delta").load(f"{BASE}/customers")

# Đọc thẳng từ CSV vì chưa có trong silver
translation = spark.read.csv(
    "/home/jovyan/work/data/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

print("Load OK")
translation.show(5)

Load OK
+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
+---------------------+-----------------------------+
only showing top 5 rows



In [7]:
from pyspark.sql import functions as F

gold_enriched = (
    order_items
    .join(orders,    "order_id")
    .join(products,  "product_id")
    .join(translation, "product_category_name", "left")
    .join(customers, "customer_id", "left")
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "product_category_name",
        F.coalesce(
            translation["product_category_name_english"],  # chỉ rõ lấy từ bảng nào
            F.col("product_category_name")
        ).alias("category_english"),
        "price",
        "freight_value",
        "order_status",
        "order_purchase_timestamp",
        "customer_state",
    )
    .filter(F.col("order_status") == "delivered")
    .filter(F.col("category_english").isNotNull())
)

gold_enriched.write.format("delta").mode("overwrite") \
    .save("/home/jovyan/work/delta_lake/gold/gold_order_items_enriched")

print("gold_order_items_enriched:", gold_enriched.count(), "rows")
gold_enriched.show(5, truncate=False)

gold_order_items_enriched: 108638 rows
+--------------------------------+--------------------------------+--------------------------------+---------------------+----------------+-----+-------------+------------+------------------------+--------------+
|order_id                        |customer_id                     |product_id                      |product_category_name|category_english|price|freight_value|order_status|order_purchase_timestamp|customer_state|
+--------------------------------+--------------------------------+--------------------------------+---------------------+----------------+-----+-------------+------------+------------------------+--------------+
|20e0101b20700188cadb288126949685|48558a50a7ba1aab61891936d2ca7681|64d0feb1bcf9c7fe7b5dad3271c10910|moveis_decoracao     |furniture_decor |89.18|16.38        |delivered   |2018-01-22 19:22:22     |MG            |
|3a830ecb5338f0ff69822d388b64822e|f8c3d249c98f91b25409df45d4a095e3|bb1865456593db5a35bcc2b23174a09d|esporte_l

In [8]:
gold_basket = (
    gold_enriched
    .groupBy("order_id")
    .agg(F.collect_set("category_english").alias("items"))  # ← đổi collect_list → collect_set
    .filter(F.size("items") >= 2)
)

gold_basket.write.format("delta").mode("overwrite") \
    .save("/home/jovyan/work/delta_lake/gold/gold_basket")

print("gold_basket:", gold_basket.count(), "đơn hàng hợp lệ")
gold_basket.show(10, truncate=False)

gold_basket: 9491 đơn hàng hợp lệ
+--------------------------------+---------------------------------------------------------------------+
|order_id                        |items                                                                |
+--------------------------------+---------------------------------------------------------------------+
|00143d0f86d6fbd9f9b38ab440ac16f5|[sports_leisure, sports_leisure, sports_leisure]                     |
|00337fe25a3780b3424d9ad7c5a4b35e|[bed_bath_table, bed_bath_table]                                     |
|003822434f91204da0a51fe4cf2aba18|[perfumery, perfumery]                                               |
|00526a9d4ebde463baee25f386963ddc|[food, food, food, food]                                             |
|008d9bf350ff02ed444b3452cf3f57e0|[toys, toys]                                                         |
|00a57dfbb049fbaae10763e2cf15f797|[computers_accessories, computers_accessories, computers_accessories]|
|00c9f7d4b0e87781465e

In [9]:
print("=== TOP 10 CATEGORIES ===")
(gold_enriched
 .groupBy("category_english")
 .count()
 .orderBy(F.desc("count"))
 .show(10, truncate=False))

print("=== PHÂN BỐ BASKET SIZE ===")
(gold_basket
 .withColumn("basket_size", F.size("items"))
 .groupBy("basket_size")
 .count()
 .orderBy("basket_size")
 .show(15))

print("=== THỐNG KÊ ===")
(gold_basket
 .withColumn("basket_size", F.size("items"))
 .select(
     F.count("order_id").alias("total_orders"),
     F.avg("basket_size").alias("avg_basket_size"),
     F.max("basket_size").alias("max_basket_size"),
 )
 .show())

=== TOP 10 CATEGORIES ===
+---------------------+-----+
|category_english     |count|
+---------------------+-----+
|bed_bath_table       |10952|
|health_beauty        |9465 |
|sports_leisure       |8428 |
|furniture_decor      |8157 |
|computers_accessories|7643 |
|housewares           |6795 |
|watches_gifts        |5857 |
|telephony            |4428 |
|garden_tools         |4267 |
|auto                 |4139 |
+---------------------+-----+
only showing top 10 rows

=== PHÂN BỐ BASKET SIZE ===
+-----------+-----+
|basket_size|count|
+-----------+-----+
|          2| 7285|
|          3| 1281|
|          4|  490|
|          5|  188|
|          6|  190|
|          7|   22|
|          8|    8|
|          9|    3|
|         10|    7|
|         11|    4|
|         12|    5|
|         13|    1|
|         14|    2|
|         15|    2|
|         20|    2|
+-----------+-----+
only showing top 15 rows

=== THỐNG KÊ ===
+------------+------------------+---------------+
|total_orders|   avg_basket

In [10]:
# Đọc lại để verify
check = spark.read.format("delta").load(
    "/home/jovyan/work/delta_lake/gold/gold_basket"
)
print("Verify gold_basket OK — rows:", check.count())

# Xem Delta log
from delta.tables import DeltaTable
dt = DeltaTable.forPath(spark, "/home/jovyan/work/delta_lake/gold/gold_basket")
dt.history(3).select("version", "timestamp", "operation").show()

Verify gold_basket OK — rows: 9491
+-------+--------------------+---------+
|version|           timestamp|operation|
+-------+--------------------+---------+
|      0|2026-05-05 17:41:...|    WRITE|
+-------+--------------------+---------+

